In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:
import json, time
from confluent_kafka import Producer

producer = Producer(conf)

# Called once per event, after the broker confirms (or rejects) it.
def delivery_report(err, msg):
    if err is not None:
        print(f"  FAILED: {err}")
    else:
        key = msg.key().decode("utf-8")
        print(f"  delivered  key={key:8}  ->  partition {msg.partition()}  offset {msg.offset()}")

TOPIC = "post-engagement"

# 5 events, ALL for the same post -> same key -> should all land in ONE partition
events = [
    {"post_id": "post-1", "type": "view", "user_id": "u10", "ts": int(time.time())},
    {"post_id": "post-1", "type": "view", "user_id": "u11", "ts": int(time.time())},
    {"post_id": "post-1", "type": "like", "user_id": "u10", "ts": int(time.time())},
    {"post_id": "post-1", "type": "view", "user_id": "u12", "ts": int(time.time())},
    {"post_id": "post-1", "type": "like", "user_id": "u12", "ts": int(time.time())},
]

print("Producing 5 events for post-1 ...")
for e in events:
    producer.produce(
        topic=TOPIC,
        key=e["post_id"],              # the KEY -> decides the partition
        value=json.dumps(e),           # the VALUE -> the event body, as JSON text
        callback=delivery_report,      # called when the broker confirms this event
    )
    producer.poll(0)                   # give the client a moment to fire callbacks

producer.flush()                       # wait until ALL events are confirmed before finishing
print("Done.")

In [0]:
import json, time
from confluent_kafka import Producer

producer = Producer(conf)   # conf comes from kafka_conn
TOPIC = "post-engagement"

def delivery_report(err, msg):
    if err is None:
        print(f"  VIEW  key={msg.key().decode():7} -> partition {msg.partition()} offset {msg.offset()}")
    else:
        print(f"  FAILED: {err}")

posts = ["post-1", "post-2", "post-3", "post-4", "post-5"]

print("Producer #1 (views): one view per post, 5 posts ...")
for p in posts:
    e = {"post_id": p, "type": "view", "user_id": "u99", "ts": int(time.time())}
    producer.produce(TOPIC, key=p, value=json.dumps(e), callback=delivery_report)
    producer.poll(0)

producer.flush()
print("Done.")

In [0]:
import json, time
from confluent_kafka import Producer

producer2 = Producer(conf)   # a SECOND, independent producer
TOPIC = "post-engagement"

def delivery_report(err, msg):
    if err is None:
        print(f"  LIKE  key={msg.key().decode():7} -> partition {msg.partition()} offset {msg.offset()}")
    else:
        print(f"  FAILED: {err}")

posts = ["post-1", "post-2", "post-3", "post-4", "post-5"]

print("Producer #2 (likes): one like per post, 5 posts ...")
for p in posts:
    e = {"post_id": p, "type": "like", "user_id": "u77", "ts": int(time.time())}
    producer2.produce(TOPIC, key=p, value=json.dumps(e), callback=delivery_report)
    producer2.poll(0)

producer2.flush()
print("Done.")